# 03 - ONNX Export

Export the trained model to ONNX format and verify inference.

In [1]:
import logging
import torch
logging.basicConfig(level=logging.INFO)

from lss_model.config import ModelConfig
from lss_model.model import LSSEmbedModel
from lss_model.export import export_to_onnx, verify_onnx

config = ModelConfig()
model = LSSEmbedModel(config)
model.load_state_dict(
    torch.load("output/lss-embedding-model/pytorch_model.bin", map_location="cpu")
)

onnx_path = export_to_onnx(model, config)
verify_onnx(onnx_path, config)

/home/cier/projects/lss/lss-model/lss_model/export.py:24: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `LSSEmbedModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `LSSEmbedModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/opt/stabilitymatrix/Data/Assets/Python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
INFO:onnx_ir.passes.common.unused_removal:No unused functions to remove
INFO:onnx_ir.passes.common.unused_removal:Removed 20 unused nodes
INFO:onnx_ir.passes.common.unused_removal:No unused functions to remove
INFO:onnxscript.optimizer._constant_folding:Skipping constant folding for node 'node_Shape_4' because it is graph input to preserve graph signature
INFO:onnxscript.optimizer._constant_folding:Skipping constant folding for node 'node_Shape_5' because it is graph input to preserve graph signature
INFO:onnxscript.optimizer._constant_folding:Skipping constant folding for node 'node_embedding' because it is graph input to preserve graph signature
INFO:onnxscript.optimizer._constant_folding:Skipping constant folding for no

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


INFO:onnxscript.optimizer._constant_folding:Skipping constant folding for node 'node_slice_5' because it is graph input to preserve graph signature
INFO:onnxscript.optimizer._constant_folding:Skipping constant folding for node 'node_slice_7' because it is graph input to preserve graph signature
INFO:onnxscript.optimizer._constant_folding:Skipping constant folding for node 'node_slice_9' because it is graph input to preserve graph signature
INFO:onnxscript.optimizer._constant_folding:Skipping constant folding for node 'node_slice_11' because it is graph input to preserve graph signature
INFO:onnx_ir.passes.common.unused_removal:Removed 157 unused nodes
INFO:onnxscript.rewriter:Applied 50 of general pattern rewrite rules.
INFO:onnx_ir.passes.common.unused_removal:No unused functions to remove
INFO:onnxscript.optimizer._constant_folding:Skipping constant folding for node 'node_Shape_4' because it is graph input to preserve graph signature
INFO:onnxscript.optimizer._constant_folding:Skippi

[torch.onnx] Optimize the ONNX graph... ✅


INFO:lss_model.export:ONNX model exported to output/lss-embedding-model/model.onnx
INFO:lss_model.export:ONNX inference OK, output shape: (1, 384)


In [2]:
import onnxruntime
import numpy as np
from transformers import AutoTokenizer

session = onnxruntime.InferenceSession(onnx_path)

texts = ["hello world", "semantic search"]
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="np")
embeddings = session.run(
    ["sentence_embedding"],
    {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "token_type_ids": inputs["token_type_ids"],
    },
)[0]

sim = embeddings @ embeddings.T
print(f"Similarity matrix:\n{sim}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


Similarity matrix:
[[0.9999999  0.84718645]
 [0.84718645 0.9999998 ]]
